<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
         <img src="../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington – ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Systems Platform Engineer</p>
    </td>
  </tr>
</table>

# CN0566 Phaser Hardware Architecture

## Introduction

Welcome to the hardware deep-dive for the **CN0566 Phaser** platform. In the previous notebook, we explored the theory of phased arrays — beam steering, array factors, tapering, and performance metrics. Now we'll see how those principles are implemented in real hardware.

![Phaser Diagram](https://analogdevicesinc.github.io/documentation/_images/blockdiagram.svg)

The CN0566 is a complete X-band (8–16 GHz) beamforming development platform that integrates:
- An **8-element receive phased array** with electronic beam steering
- **Per-channel phase and amplitude control** for tapering and beam shaping
- **Frequency synthesis and mixing** to interface with standard SDR hardware
- **Real-time control** via embedded Linux on Raspberry Pi

**What You'll Learn:**
- How the CN0566 implements phased array principles in hardware
- Signal flow from antenna to digital samples
- The role of each component (ADAR1000, ADF4159, PlutoSDR, etc.)
- System specifications and performance characteristics

**Key Capabilities:**

- **Frequency Range:** 8–16 GHz (X-band) at antenna, 2.2 GHz IF to PlutoSDR
- **Beam Steering:** ±60° electronic steering with 2.8° phase resolution
- **Gain Control:** 31 dB range per channel with 0.5 dB resolution
- **Array Configuration:** 8 elements, λ/2 spacing (~14 mm at 10.3 GHz)
- **Applications:**
  - CW beam steering and direction finding
  - FMCW radar and target tracking
  - Beamforming algorithm development and calibration
  - Phased array education and training

## Details

The phaser kit is composed of the following components:
- Raspberry Pi running Kuiper Linux
- Pluto SDR module based on:
  - __[AD9363](https://www.analog.com/en/products/ad9363.html)__ — RF Agile Transceiver
  - Zynq SoC (also running Linux)
- Custom phaser array board consisting of:
  - __[ADL8107](https://www.analog.com/en/products/adl8107.html)__ ×8 — GaAs, pHEMT, MMIC, Low Noise Amplifier, 6 GHz to 18 GHz
  - __[ADAR1000](https://www.analog.com/en/products/adar1000.html)__ ×2 — 8–16 GHz, 4-channel beamformer
  - __[ADF4159](https://www.analog.com/en/products/adf4159.html)__ — Fractional-N frequency synthesizer
  - __[LTC5548](https://www.analog.com/en/products/ltc5548.html)__ — 2–14 GHz mixer
  - __[AD7291](https://www.analog.com/en/products/ad7291.html)__ — 8-channel I²C SAR ADC with temperature sensor

## System Specifications

| Parameter | Value | Notes |
|-----------|-------|-------|
| **Frequency** | | |
| RF Operating Range | 8–16 GHz | X-band, ADAR1000 range |
| Typical Operating Frequency | 10.3 GHz | Used in tutorials |
| IF Frequency | 2.2 GHz | To PlutoSDR |
| LO Range | 10.5–12.7 GHz | ADF4159 + HMC735 VCO |
| **Array Configuration** | | |
| Number of Elements | 8 | (32 patches: 4 rows × 8 columns) |
| Element Spacing | ~14 mm | λ/2 at 10.3 GHz |
| Array Aperture | ~98 mm | 7 × 14 mm spacing |
| Physical Antenna | 32 patches | 8 columns × 4 rows |
| **Beamforming** | | |
| Phase Resolution | 2.8° | 360° / 128 steps |
| Gain Range | 31 dB | Per channel |
| Gain Resolution | 0.5 dB | Per channel |
| Steering Range | ±60° | Practical limit |
| **Performance** | | |
| LNA Gain | 24 dB | ADL8107 |
| System SFDR | ~56 dBc | At IF output |
| **Digital Interface** | | |
| Sample Rate | Up to 61.44 MSPS | PlutoSDR (AD9363) |
| Bandwidth | Up to 20 MHz | Instantaneous |
| Resolution | 12-bit | ADC/DAC |
| **Control** | | |
| Embedded Controller | Raspberry Pi 4 | Kuiper Linux |
| Beamformer Interface | SPI | To ADAR1000 |
| Host Interface | Ethernet/USB | IIO framework |

## Signal Flow: Receive Path

Understanding the signal flow is critical to using the Phaser effectively. Here's the complete receive path from antenna to digital samples:

### Receive Chain (Antenna → Digital)

<img src="resources/signal-flow-rx.drawio.svg" width="600" alt="RX Signal Flow"/>

### Key Points:

1. **RF to IF Conversion:** The mixer translates the 10.3 GHz signal down to 2.2 GHz so the PlutoSDR can digitize it (AD9363 operates up to 3.8 GHz)

2. **Beamforming Happens at RF:** Phase and gain adjustments occur at 10.3 GHz before downconversion, preserving the beam pattern

3. **LO Frequency Planning:** 
   - For 10.3 GHz signal: LO = 10.3 GHz + 2.2 GHz = **12.5 GHz**
   - General rule: LO = RF_signal + IF_frequency

4. **Coherent Summation:** All 8 channels are phase-aligned and summed by the ADAR1000 before entering the single mixer/digitizer chain

Let's take a deeper look at each element in the receive chain:

## Receive Antenna

The receive antenna consists of a total of 32 antenna patches. However, each column of 4 patches is summed equally via printed circuit board (PCB) trace Wilkinson splitters. This yields 8 effective antenna elements.
<div align="center">
<img src="resources/antenna-column.svg" width="400" align="centre">
<img src="resources/rx-antenna.svg" width="400">
</div>
Each of these combined group of 4 elements will have a higher gain, and narrower beam, than any one individual antenna patch. Also built into the antenna is a quarter-wavelength shorting stub to provide some electrostatic discharge (ESD) protection. Each element is AC coupled, via 10 nF capacitors, to the 24 dB gain ADL8107 low noise amplifier. The LNAs increase the sensitivity of the array, providing a sharp beam pattern even with low power microwave sources.

The output from the antennas is fed into the beamformers.

## Beamformers
At the center of the RF board is a pair of ADAR1000 4-channel, 8 GHz to 16 GHz, beamforming ICs. The ADAR1000 allows per-channel, 360° phase adjustment with 2.8° resolution, and 31 dB gain adjustment with 0.5 dB resolution. The two ADAR1000s are capable of bidirectional, half-duplex operation; however, the CN0566 only connects the ADAR1000 receive paths. The outputs of the ADL8107 LNAs are phase and amplitude shifted by an ADAR1000, and then summed together at its RFIO output as shown below.

<div align="center">
<img src="resources/adar1000-working.png" alt="ADAR1000" width="800">
</div>

Each output of the beamformers is connected to the input of the mixer.

## Mixers and Filtering

The RFIO output of the ADAR1000 passes through a 10.6 GHz low pass filter and then enters the RF port of the LTC5548 mixer.

<div align="center">
<img src="resources/mixers.svg" alt="Mixers" width="600">
</div>

The low pass filter removes the high-side image of the mixer (which in the figure above will appear at 12.2 GHz + 2.2 GHz = 14.4 GHz) as well as any reradiation of the LO (12.2 GHz). The LTC5548 mixer outputs an IF of 2.2 GHz, which is filtered by a 2.5 GHz low pass filter.

The figure below shows the measured results of the receive signal path (ADL8107 + ADAR1000 + 10.62 GHz low pass filter + LTC5548 + 2.5 GHz low pass filter). This is for an LO of 12.2 GHz, antenna input of 10 GHz, and an IF of 2.2 GHz. Note that the 12.2 GHz and the 14.4 GHz will be further attenuated by the input bandwidth of the PlutoSDR, resulting in an SFDR of approximately 56 dBc as shown by markers M1 to M4 (-23 dBm + 79 dBm = 56 dBc).

<div align="center">
<img src="resources/sfdr-rx1.svg" alt="Mixers" width="600">
</div>


### Local Oscillator Synthesizer
The ADF4159 phase-locked loop (PLL) and the HMC735 voltage-controlled oscillator (VCO) combine to form a frequency synthesizer with a range of 10.5 GHz to 12.7 GHz as shown below. This signal is used to drive the LO port of all the mixers. For communications and other fixed-frequency applications, the LO frequency is typically set to 2.2 GHz above the desired signal at the antenna. Therefore, the LO is generally between 12.2 GHz and 12.7 GHz. The ADF4159 is also capable of generating FMCW ramps or “chirps” for radar applications. The ADF4159 includes a variety of chirp ramp rates and shapes including sawtooth, triangular, and parabolic.

<div align="center">
<img src="resources/lo-architecture.png" alt="LO Architecture" width="600">
</div>

### LO Generation
Alternatively, the on-board synthesizer can be disabled, and an external LO can be applied to the LO input SMA connector. This allows the Phaser to be synchronized to an external radio, or several Phaser boards to be synchronized to a single LO. Whether generated on-board or externally, the local oscillator is split using monolithic microwave integrated circuit (MMIC) splitter or combiners to the two receive mixers, and to either the on-board transmit path or to an LO output port.



## PlutoSDR: The Digital Interface

The PlutoSDR is a compact software-defined radio platform built around the [**AD9363 RF Agile Transceiver**](https://www.analog.com/en/products/ad9363.html) and an embedded Zynq SoC from Xilinx. Within the CN0566 system, the PlutoSDR acts as the core **digitizer** — meaning it is responsible for converting high-frequency RF signals into digital data (and vice versa), enabling them to be processed in software.

The term *digitizer* refers to this conversion process, where analog signals from the RF domain are sampled by high-speed ADCs and later re-synthesized for transmission using DACs. This makes the PlutoSDR an essential bridge between the RF front-end and digital signal processing domain.

The **AD9363** at the heart of PlutoSDR is a fully integrated transceiver that supports:
- RF tuning from 325 MHz to 3.8 GHz
- Up to 20 MHz of instantaneous bandwidth
- Dual-channel Tx and Rx operation
- Built-in analog and digital filters, LO synthesis, and AGC

The Zynq SoC on the Pluto board integrates a dual-core ARM processor and programmable logic, allowing local signal processing, configuration, and communication over USB or Ethernet using the IIO framework. This SoC also hosts the `iiod` daemon if configured, allowing remote access to the AD9363's controls and buffers.

In the CN0566 system, the PlutoSDR performs the following roles:
- **Receiver (Rx):** Captures downconverted IF signals (typically centered around 2.2 GHz) from the front-end mixer (e.g., LTC5548).
- **Transmitter (Tx):** Optionally used for signal injection or beam verification through upconverted output.
- **Clock source:** Supplies or locks to external clocks if required for synchronization.
- **IIO streaming interface:** Provides real-time buffered I/Q data to host applications (e.g., this notebook).

Together, the AD9363 and Zynq SoC provide a full bits-to-RF signal chain, abstracted through the **pyADI-IIO** interface in Python. This allows users to configure LO frequency, bandwidth, gain modes, buffer sizes, and more through high-level code, without needing to directly manage hardware registers.

<img src="resources/ad9361.png" alt="AD9361 Block Diagram" width="800"/>

## Raspberry Pi: System Controller

In the CN0566 Phaser system, the **Raspberry Pi** acts as the central controller that configures, synchronizes, and manages all beamforming operations. It is the "brain" of the system, orchestrating all hardware components.


### Key Responsibilities

**1. Beamformer Control**
- Communicates with ADAR1000 beamformer ICs via **SPI** interface
- Programs phase and gain settings for each of the 8 channels
- Triggers beam updates and handles system initialization
- Manages GPIO signals for control and synchronization

**2. System Integration**
- Runs the **IIO (Industrial I/O) subsystem** for device abstraction
- Hosts the `iiod` daemon, exposing the CN0566 as a network-accessible device
- Provides the **network interface** (Ethernet) for remote access
- Manages communication between PlutoSDR and host PC

**3. Software Stack**
- Runs **Kuiper Linux** — a specialized Analog Devices Linux distribution
- Includes pre-installed drivers for ADAR1000, ADF4159, and PlutoSDR
- Supports **pyADI-IIO** library for Python-based control
- Enables remote Jupyter Notebook access and scripting


### Software Architecture

```
[Host PC - Jupyter Notebook]
        ↓ (Ethernet/USB)
        ↓
[Raspberry Pi - Kuiper Linux]
   ├── iiod (IIO daemon)
   ├── ADAR1000 driver (SPI)
   ├── ADF4159 driver (SPI)
   └── PlutoSDR interface (USB)
```

When you run Python code in a Jupyter Notebook, the commands are sent over the network to the Raspberry Pi, which then:
1. Translates high-level commands (e.g., "steer beam to 30°") into register writes
2. Sends SPI transactions to configure ADAR1000 phase/gain values
3. Streams captured data from PlutoSDR back to the host PC

Essentially, the Raspberry Pi turns the CN0566 hardware into a **programmable, scriptable phased-array appliance**.


## External Signal Source: HB100

While the Phaser kit does have a built-in transmitter, we're not going to use it for this workshop. Instead, we're using the **HB100** module as an external X-band signal source.

---

### What is the HB100?

The HB100 is a simple, low-cost X-band Doppler radar module operating around **10.525 GHz**. It's included with the Phaser kit and serves as a convenient continuous-wave (CW) signal source for beamforming experiments.

**Key Features:**
- **Frequency:** ~10.2–10.6 GHz (typically 10.525 GHz, varies by unit)
- **Output Power:** ~10–15 mW (10–12 dBm)
- **Power Supply:** 3–5V DC (typically 3× AA batteries in series = 4.5V)
- **Form Factor:** Small PCB module with microstrip patch antenna
- **Purpose:** Provides a stable, known RF source for direction finding and beam steering demos

More information: [DFRobot HB100 Product Page](https://www.dfrobot.com/product-1403.html?srsltid=AfmBOoqxvUziIfRxUZjQMj95TpL07p1cqWjNkIDSNVv9eDUqzaD3e4KP)

---

### Why Use an External Source?

Using an external signal source like the HB100 provides several advantages:

1. **Known Direction:** You can physically position the HB100 at a specific angle to verify beam steering
2. **Frequency Reference:** Helps calibrate the LO frequency of the Phaser
3. **Isolation:** Separates transmit and receive, avoiding direct coupling issues
4. **Simplicity:** No need to configure transmit path — just point and detect
5. **Safety:** Low power output is safe for lab/classroom use

---

### Setup

To use the HB100:

1. **Power:** Connect 3× AA batteries (4.5V) or a 5V bench supply to the power pins
2. **Position:** Place the HB100 1–3 meters away from the Phaser antenna, pointing toward it
3. **Orientation:** Ensure the HB100's patch antenna faces the Phaser array
4. **Angle:** Start at mechanical boresight (0°), then move to test steering

The HB100's frequency will be slightly different for each unit (manufacturing variation), so you'll need to measure it during setup — we'll do this in the programming notebook.